In [1]:
import pandas as pd
import pickle
import torch
from itertools import product
import networkx as nx



#fname_base = 'D:\\PhD\\Openalex\\subset_ok\\years\\2019-2022\\DyHANE\\' 
fname_base = 'DyHANE/'
base = '2019-2021/'
new = '2022/'

edge_types = ['AP', 'AI', 'PP']

Each event eij = (i, xi, j, xj, r, xij, s=1, c) 

We have a df for each relation tipe. Each row (edge) needs to be categorized into weak or strong according to the adjacency matrix of that type (category c=0 if weak, c=1 if strong).

N.B.: i mapping sono salvati nel formato index_id --> openalex_id. 

In [42]:
# STEP 1: MAPPING

#get the mapping in the form openalex_id -->index_id
def open_mapping(pckl):
    file = open(pckl, 'rb')
    diz = pickle.load(file)
    reversedDict = {val: key for (key, val) in diz.items()}
    return reversedDict


#get the mapped id
#mapping is in the form: openalex_id --> index
def find_mapping(id, mapping):
    max_id = max(mapping.values())
    index = mapping[id] if id in mapping else max_id+1
    mapping[id] = index #true update only if index > max_id
    return index


#update the mapping dictionary in the form index_id --> openalex_id
def update_mapping(pckl, mapping):
    reversedDict = {val: key for (key, val) in mapping.items()}
    with open(pckl, 'wb') as fp:
        pickle.dump(reversedDict, fp)



In [5]:
# STEP 1: MAPPING

mapping_A = open_mapping(fname_base+'mapping_A.pkl')
mapping_P = open_mapping(fname_base+'mapping_P.pkl')
mapping_I = open_mapping(fname_base+'mapping_I.pkl')


# for each edge_type, for each edge of that type, for each node incident on the edge
#   get its mapping (raw if already existing, new otherwise). Update mapping if needed
#   if new, update mapping and compute x

df_events = pd.DataFrame(columns=['openalex_id_i', 'openalex_id_j', 'id_i', 'id_j', 'r_type']) #'x_ij', 'category'

for etype in edge_types:
    df = pd.read_csv(fname_base+new+etype+'.csv')
    if etype == 'AP':
        df_ev = df[['author', 'paper']]
        df_ev.rename(columns={'author': 'openalex_id_i', 'paper':'openalex_id_j'}, inplace=True)
        df_ev['id_i'] = df_ev.apply(lambda row: find_mapping(row['openalex_id_i'], mapping_A), axis=1)
        df_ev['id_j'] = df_ev.apply(lambda row: find_mapping(row['openalex_id_j'], mapping_P), axis=1)
    elif etype == 'AI':
        df_ev = df[['author_id', 'institution_id']]
        df_ev.rename(columns={'author_id': 'openalex_id_i', 'institution_id':'openalex_id_j'}, inplace=True)
        df_ev['id_i'] = df_ev.apply(lambda row: find_mapping(row['openalex_id_i'], mapping_A), axis=1)
        df_ev['id_j'] = df_ev.apply(lambda row: find_mapping(row['openalex_id_j'], mapping_I), axis=1)
    else: #etype == 'PP'
        df_ev = df[['citing', 'cited']]
        df_ev.rename(columns={'citing': 'openalex_id_i', 'cited':'openalex_id_j'}, inplace=True)
        df_ev['id_i'] = df_ev.apply(lambda row: find_mapping(row['openalex_id_i'], mapping_P), axis=1)
        df_ev['id_j'] = df_ev.apply(lambda row: find_mapping(row['openalex_id_j'], mapping_P), axis=1)
    df_ev['r_type'] = etype
    df_events = pd.concat([df_events, df_ev], axis=0)

df_events.to_csv(fname_base+new+'events_edges.csv', index=False)

update_mapping(fname_base+'mapping_A.pkl', mapping_A)
update_mapping(fname_base+'mapping_P.pkl', mapping_P)
update_mapping(fname_base+'mapping_I.pkl', mapping_I)

C:\Users\Liliana\AppData\Local\Temp\ipykernel_11192\1645107016.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ev.rename(columns={'author': 'openalex_id_i', 'paper':'openalex_id_j'}, inplace=True)
C:\Users\Liliana\AppData\Local\Temp\ipykernel_11192\1645107016.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ev.rename(columns={'citing': 'openalex_id_i', 'cited':'openalex_id_j'}, inplace=True)


In [ ]:
#STEP 2: FEATURES UPDATE

#vedi remapping_vero

#return the row to be replaced/added to feature tensor X
def get_attributes(id, df):
    attrs = df[df['id'] == id]
    return attrs
    
#def update_attributes(X)

In [3]:
AX = pd.read_csv(fname_base+'2019-2021\\features\\AX.csv')
AX.head(5)

,aid,id,n_works,n_cit,impact_factor,h_index,i10_index,name_encoded
0,0,A5009027337,0.674145,2.286834,0.491801,1.745403,0.568889,"[0.022931719198822975, 0.003456080798059702, -..."
1,1,A5086198262,4.888428,34.804942,1.942068,8.694262,7.577777,"[-0.03792957216501236, 0.1179778128862381, 0.0..."
2,2,A5077097325,0.682328,3.057577,-0.516099,1.869490,0.812788,"[-0.03513007611036301, -0.012318928726017475, ..."
3,3,A5079098245,0.915545,3.147933,0.783342,2.241750,1.095198,"[-0.003983520902693272, 0.055438704788684845, ..."
4,4,A5042195176,0.833715,0.430940,-0.448584,1.124969,0.479032,"[-0.02594980224967003, 0.0600975826382637, 0.0..."


In [ ]:
#STEP 2: FEATURES UPDATE

AX = torch.load(fname_base+base+'AX_tensor.pt')
AY = torch.load(fname_base+base+'AY_tensor.pt')
PX = torch.load(fname_base+base+'PX_tensor.pt')
IX = torch.load(fname_base+base+'IX_tensor.pt')


####vedi remapping vero (2022)

In [37]:
#STEP PRELIMINARE: generare mp based graphs

#get the meta-path-based graph for a specific edge type
def get_metapath_graph(df, source_col, target_col, graph_type): #graph_type is nx.Graph or nx.DiGraph
    return nx.from_pandas_edgelist(df, source=source_col, target=target_col, create_using=graph_type) #edge_key=key_col, edge_attr=weigth_col

#get the (sparse) adjacecy matrix of a meta-path-based graph
def get_adjacency_matrix(G):
    return nx.adjacency_matrix(G)



mp_graphs = []
AA = pd.read_csv(fname_base+base+'AA.csv') #source - target - type - weight

#APA
APA = AA[AA['type'] == 'APA']
G_APA = get_metapath_graph(APA, 'source', 'target', nx.Graph)
mp_graphs.append(G_APA)
#Adj_APA = get_adjacency_matrix(G_APA)

#AIA
AIA = AA[AA['type'] == 'AIA']
G_AIA = get_metapath_graph(AIA, 'source', 'target', nx.Graph)
mp_graphs.append(G_AIA)
#Adj_AIA = get_adjacency_matrix(G_AIA)

#APPA
APPA = AA[AA['type'] == 'APPA']
G_APPA = get_metapath_graph(APPA, 'source', 'target', nx.DiGraph) #this graph is directed
mp_graphs.append(G_APPA)
#Adj_APPA = get_adjacency_matrix(G_APPA)

In [38]:
AA = pd.read_csv(fname_base+base+'AA.csv') #source - target - type - weight
AA.head(5)

,source,target,type,weight
0,A5000004709,A5025012438,APA,1
1,A5000004709,A5035731144,APA,1
2,A5000004709,A5060200216,APA,1
3,A5000014798,A5031305137,APA,1
4,A5000014798,A5072420283,APA,1


In [39]:
#STEP 3: WEAK/STRONG CATEGORIZATION

def metapaths_extraction(df, id_i, id_j, etype): #x1 = p1 or x1 = i1
    df_t = df[df['r_type'] == etype]
    df_t = df_t[['openalex_id_i', 'openalex_id_j']]
    
    if etype == 'AP' or etype == 'AI':
        df_sub = df_t[(df_t['openalex_id_j'] == id_j) & (df_t['openalex_id_i'] != id_i)]
        authors = df_sub['openalex_id_i'].tolist()
        return [[id_i, a] for a in authors]
    else: #etype == PP, df = AP
        df_t = df[df['r_type'] == 'AP']
        df_t = df_t[['openalex_id_i', 'openalex_id_j']]
        df_sub_l = df_t[df_t['openalex_id_j'] == id_i]
        authors_l = df_sub_l['openalex_id_i'].tolist()
        df_sub_r = df_t[df_t['openalex_id_j'] == id_j]
        authors_r = df_sub_r['openalex_id_i'].tolist()
        return list(product(authors_l, authors_r))
    



#assign weak/strong label (category) to edge. #1 --> strong, 0 --> weak (no mps or entry already existing)
def check_entry1(graphs, mps):
    #if mps.str.len() == 0:
    if len(mps) == 0:
        return 0
    #return int(not any(check_entry2(graphs, pair[0], pair[1]) for pair in mps)) #ret 1 if nessuno la ha, cioè any=False, so not any = True
    #versione 2, all metapaths present to be weak
    return int(not all(check_entry2(graphs, pair[0], pair[1]) for pair in mps)) #set tutti presenti, alora vicinato stabile weak 0

#check if authors pair already exists in one of the meta-path-based graphs
def check_entry2(graphs, id_a0, id_a1):
    return any(check_entry_G(G, id_a0, id_a1) for G in graphs) #True se esiste almeno 1 mp di qualche tipo tra quegli autori
   

#check if authors pair already exists in the specified meta-path-based graph
def check_entry_G(G, id_a0, id_a1):
    return G.has_edge(id_a0, id_a1) #true se il grafo ha quell'arco


##### alternativa:
#



In [40]:
#STEP 3: WEAK/STRONG CATEGORIZATION

# per ogni arco, genera nuovi meta-paths
# per ogni MP, check adjacency matrices: if already 1, c=0 (weak); otherwise c=1 and adj[i][j] = 1


df_events = pd.read_csv(fname_base+new+'events_edges.csv')
print(df_events.shape)

df_events['mps'] = df_events.apply(lambda row: metapaths_extraction(df_events, row['openalex_id_i'], row['openalex_id_j'], row['r_type']), axis=1)
print(df_events.shape)

#df_events = df_events[df_events['mps'].str.len() > 0]
#print(df_events.shape)
df_events['category'] = df_events.apply(lambda row: check_entry1(mp_graphs, row['mps']), axis=1) #1 strong, 0 weak
print(df_events.shape)


df_AA = df_events[df_events['mps'].str.len() > 0].explode(column='mps')
print(df_AA.shape)

df_AA[['source', 'target']] = pd.DataFrame(df_AA['mps'].tolist(), index=df_AA.index)
#df_AA.drop(columns=['', ''], inplace=True)
print(df_AA.shape)
df_AA.rename(columns={'r_type':'type'}, inplace=True)
df_AA['weight'] = 1
df_AA = df_AA[['source', 'target', 'type', 'weight']]
df_AA = df_AA.groupby(['source','target', 'type'])['weight'].sum().reset_index()
df_AA.to_csv(fname_base+new+'AA_new.csv', index=False) 
print(df_AA.shape)


AA_old = pd.read_csv(fname_base+base+'AA.csv') #source - target - type - weight
AA_all = pd.concat([AA_old, df_AA], axis=0)
#AA = AA_old.groupby(['source','target']).size().reset_index(name='count')
# Use GroupBy() & compute sum on specific column
AA_all = AA_all.groupby(['source','target', 'type'])['weight'].sum().reset_index()
AA_all.to_csv(fname_base+new+'AA_all.csv', index=False)
print(AA_all.shape)
            

df_events = df_events.drop('mps', axis=1)
print(df_events.shape)
df_events.to_csv(fname_base+new+'events_edges_cat.csv', index=False)


(3539, 5)
(3539, 6)
(3539, 7)
(7212, 7)
(7212, 9)
(6938, 4)
(56772, 4)
(3539, 6)


In [6]:
df_events['category'].value_counts() #vs 2064-119 any

category
1    2135
0    1404
Name: count, dtype: int64

In [ ]:
#STEP 4: INFLUENCED NODE SET

#per ogni evento, if c=1 add also its neighbors to I

In [7]:
''''1.1) for each edge:
        
        1.1.2) check weak/strong (AA already exists in one of raw G_type?). 
        Add (if not already present) AA to new AA_type.csv and mark event category c as 0/1 
        [eij = <id_i, id_j, r, attr_ij, c>]'''

#save new meta-paths (pairs of target nodes) for each edge type
def metapaths_extraction(mp_graphs): #e.g., 'AP'

    df_events = pd.DataFrame(columns=['id_i', 'id_j', 'r_type', 'x_ij', 'category'])

    for etype in edge_types:
        df = pd.read_csv(fname_base+new+etype+'.csv')
        if etype == 'AP':
            df.drop(columns='position', inplace=True)
            author_col = 'author'
            other_col = 'paper'
            df['mps'] = df.apply(lambda row: metapaths_extraction(df, author_col, other_col, row[author_col], row[other_col])) 

        elif etype == 'AI':
            author_col = 'author_id'
            other_col = 'institution_id'
            df['mps'] = df.apply(lambda row: metapaths_extraction(df, author_col, other_col, row[author_col], row[other_col])) 

        else: #etype == 'PP'
            df.drop(columns='year', inplace=True)
            df_AP = pd.read_csv(fname_base+new+'AP.csv')
            df['mps'] = df.apply(lambda row: metapaths_extraction(df_AP, row['citing'], row['cited']))

        #initialize df of events
        df_ev = df
        df_ev.rename(columns={df_ev.columns[0]:'id_i', df_ev.columns[1]:'id_j'})
        df_ev['r_type'] = etype
        df_ev['category'] = df_ev.apply(lambda row: check_entry(mp_graphs, row['mps'])) #1 or 0 

        df_AA = df.explode(column='mps')
        df_AA[['source', 'target']] = pd.DataFrame(df_AA['mps'].tolist(), index=df_AA.index)
        #df_AA.drop(columns=['', ''], inplace=True)
        df_AA = df_AA[['source', 'target']]
        df_AA['type'] = etype
        df_AA['weight'] = 1
        df_AA.to_csv(fname_base+new+'AA_from'+etype+'_new.csv') 
        
            
            
#get pairs of authors for individual edge of type AP or AI       
def metapaths_extraction(df, author_col, other_col, a1, x1): #x1 = p1 or x1 = i1
    df = df[[author_col, other_col]]
    df_sub = df[(df[other_col] == x1) & (df[author_col] != a1)]
    authors = df_sub[author_col].tolist()
    return [[a1, a] for a in authors]

#get pairs of authors for individual edge of type PP
def metapaths_extraction(df, p1, p2):
    df = df[['author', 'paper']]
    df_sub_l = df[df['paper'] == p1]
    authors_l = df_sub_l['author'].tolist()
    df_sub_r = df[df['paper'] == p2]
    authors_r = df_sub_r['author'].tolist()
    return list(product(authors_l, authors_r))

#get the meta-path-based graph for a specific edge type
def get_metapath_graph(df, source_col, target_col, graph_type): #graph_type is nx.Graph or nx.DiGraph
    return nx.from_pandas_edgelist(df, source=source_col, target=target_col, create_using=graph_type) #edge_key=key_col, edge_attr=weigth_col

#get the (sparse) adjacecy matrix of a meta-path-based graph
def get_adjacency_matrix(G):
    return nx.adjacency_matrix(G)

#assign weak/strong label (category) to edge
def check_entry(graphs, mps):
    return int(any(check_entry(graphs, pair[0], pair[1]) for pair in graphs))

#check if authors pair already exists in one of the meta-path-based graphs
def check_entry(graphs, id_a0, id_a1):
    return any(check_entry_G(G, id_a0, id_a1) for G in graphs) 
   

#check if authors pair already exists in the specified meta-path-based graph
def check_entry_G(G, id_a0, id_a1):
    return G.has_edge(id_a0, id_a1)



#update meta-path based graphs (raw + new)
def update_metapaths():
    AA_old = pd.read_csv(fname_base+base+'AA.csv') #source - target - type - weight
    for etype in edge_types:
        AA_new = pd.read_csv(fname_base+new+'AA_from'+etype+'_new.csv')
        AA_old = pd.concat([AA_old, AA_new], axis=0)
        #AA = AA_old.groupby(['source','target']).size().reset_index(name='count')
        # Use GroupBy() & compute sum on specific column
        AA = AA_old.groupby(['source','target', 'type'])['weight'].sum().reset_index()
        AA.to_csv(fname_base+new+'AA_all_from'+etype+'_new.csv')



    

3 directories: old, new, old-new. 

1) generate old mp based graphs
2) for each relation type 
    1.1) for each edge:
        1.1.1) extract MPS (pairs AA)
        1.1.2) check weak/strong (AA already exists in one of old G_type?). Add (if not already present) AA to new AA_type.csv and mark event category c as 0/1 
        [eij = <id_i, id_j, r, attr_ij, c>]
        1.1.3) for each node:
            - get mapping
            - get attributes/label
            - process attributes/label
        1.1.4) compute edge mapping and process attributes
        [eij = <i, j, xi, xj, r, xij, s=1, c>]
    1.2) save new etype_map.csv
    1.3) save new AA_type.csv
    1.4) generate new mp based graph 
3) create unique new file with all events
4) update node attributes matrices X and save in old-new
5) create empty influenced node set I
6) for each event:
    6.1) add i and j (with types) to I
    6.2) if c == 1 add N(i) and N(j). If i is of target type, add to I also N_mp(i) according to all new G.
7) create subset of graph, including nodes in I and all existing edges connecting them
8) prepare file for future updates (concat):
    - AA
    - edges
    - ...

In [18]:
#check num nodi

AP_base = pd.read_csv(fname_base+base+'AP.csv')
print('A 2019-2021:', AP_base['author'].drop_duplicates().shape[0])

AP_new = pd.read_csv(fname_base+new+'AP.csv')
print('A 2022:', AP_new['author'].drop_duplicates().shape[0])

authors_base = AP_base['author'].drop_duplicates().tolist()
AP_x = AP_new[~AP_new['author'].isin(authors_base)]
print('A 2022 not in 2019-2021:', AP_x['author'].drop_duplicates().shape[0])

A 2019-2021: 13116
A 2022: 2584
A 2022 not in 2019-2021: 2281


In [20]:

I = pickle.load(open(fname_base+new+'I.pkl', 'rb'))
authors_I = I['authors']

print(len(authors_I))

x = [a for a in authors_I if a not in AP_x['author'].tolist()]
print(len(x))

x2 = [a for a in authors_I if a in AP_x['author'].tolist()]
print(len(x2))

2005
2005
0


In [17]:
#quindi I contiene 2005 nodi A, nessuno dei quali è del 2022

#vediamo se il problema è in events o solo nella generazione di I 

df_ev = pd.read_csv(fname_base+new+'events_edges_cat.csv')
df_ev_AP = df_ev[df_ev['r_type'] == 'AP']
print(df_ev_AP.shape[0])

df_events = pd.read_csv(fname_base+new+'events_edges.csv')
df_events_AP = df_events[df_events['r_type'] == 'AP']
print(df_events_AP.shape[0])

print(AP_new.shape[0])

2748
2748
2748


In [ ]:
#quindi il problema non è più in edges_category. Va sistemato I sul precision-tag

In [57]:
#se voglio prendere i meta-path + frequenti:

AA = pd.read_csv(fname_base+new+'AA_all_map.csv')
print(AA.shape)
AA_top = AA[AA['weight']>1]
print(AA_top.shape)

I = pickle.load(open(fname_base+new+'I_all_only.pkl', 'rb'))
authors_I = I['authors']
print('num authors in I', len(authors_I))
authors_mp = list(set(AA_top['source'].drop_duplicates().tolist() + AA_top['target'].drop_duplicates().tolist()))
print('num authors from MP', len(authors_mp))
authors = list(set(authors_I+authors_mp))
print('authors tot new', len(authors))
I['authors'] = authors
with open(fname_base+new+'I.pkl', 'wb') as fp:
        pickle.dump(I, fp)


(56772, 4)
(6448, 4)
num authors in I 3409
num authors from MP 1368
authors tot new 4420


In [64]:
#Non basta, i vicini degli a in AA vanno già messi in I.

#serve il remapping degli edges. Per i nuovi ok, sono id_i e id_j di df_events.
#I vecchi li abbiamo già

def one_hop_neighborhood(id, AP_new, AP_old, AI_new, AI_old, AA): #all mapped
    nei_A = []
    nei_I = []
    nei_P = []

    AP_old_sub = AP_old[AP_old['author'] == id]
    AP_new_sub = AP_new[AP_new['id_i'] == id]
    nei_P = list(set(AP_old_sub['paper'].tolist() + AP_new_sub['id_j'].tolist()))
    AI_old_sub = AI_old[AI_old['author_id'] == id]
    AI_new_sub = AI_new[AI_new['id_i'] == id]
    nei_I = list(set(AI_old_sub['institution_id'].tolist() + AI_new_sub['id_j'].tolist()))
    AA_sub = AA[AA['source'] == id]
    nei_A = AA_sub['target'].drop_duplicates().tolist()
 
    return nei_A, nei_I, nei_P   



#AP
AP_old = pd.read_csv(fname_base+base+'AP_map.csv')
AP_new = df_events[df_events['r_type'] == 'AP']
AP_new = AP_new[['id_i', 'id_j']]

#AI
AI_old = pd.read_csv(fname_base+base+'AI_map.csv')
AI_new = df_events[df_events['r_type'] == 'AI']
AI_new = AI_new[['id_i', 'id_j']]

#AA all
AA = pd.read_csv(fname_base+new+'AA_all_map.csv')



MP = {}
MP['authors'] = []
MP['papers'] = []
MP['institutions'] = []

for id in authors:
    nei = one_hop_neighborhood(id, AP_new, AP_old, AI_new, AI_old, AA)
    MP['authors'].extend(nei[0])
    MP['institutions'].extend(nei[1])
    MP['papers'].extend(nei[2])
   

    


In [66]:
print(len(list(set(MP['authors']))))
print(len(list(set(MP['papers']))))
print(len(list(set(MP['institutions']))))

4612
3385
901


In [68]:
#merge 
all_authors = list(set(I['authors']+MP['authors']))
all_institutions = list(set(I['institutions']+MP['institutions']))
all_papers = list(set(I['papers']+MP['papers']))

print(len(all_authors))
print(len(all_papers))
print(len(all_institutions))

I['authors'] = all_authors
I['institutions'] = all_institutions
I['papers'] = all_papers

with open(fname_base+new+'I.pkl', 'wb') as fp:
        pickle.dump(I, fp)

5454
3401
901
